# EnergyAware AI - Week 3: Hidden-State Probe Router (final router experiment)

**Why this experiment:** text features and sentence embeddings gave almost no signal (AUC about 0.5-0.7). A stronger idea from the literature is to let a *small LLM read the query* and use its internal hidden state as the router's features. This costs only one **prefill pass** of the 0.5B model (no generation), so it is cheap. We **measure that overhead in joules and add it to every routed query**, so the comparison is honest.

**Stop rule:** run this once. If the router still does not clearly beat the best static policy (see the bootstrap confidence intervals at the end), stop trying to rescue it and report the result as a measurement study.

**Needs a GPU runtime.** Do not run `pip install -U` on Colab (it caused the sympy error before).
Run from the top. Features are cached in your Drive folder, so a re-run skips the GPU step.

In [ ]:
!pip -q install nvidia-ml-py

In [ ]:
import os, re, json, time, threading, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore")
pd.set_option("display.width", 220); pd.set_option("display.max_columns", None)

SEED = 42
ORDER = ["small", "medium", "large"]
PROBE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
BATCH = 8

OUT_DIR = globals().get("OUT_DIR")
if OUT_DIR is None:
    if os.path.exists("/kaggle/working"):
        OUT_DIR = "/kaggle/working/energyaware"
    else:
        try:
            from google.colab import drive
            drive.mount("/content/drive")
            OUT_DIR = "/content/drive/MyDrive/energyaware"
        except Exception:
            OUT_DIR = "./energyaware"
RESULTS_PATH = f"{OUT_DIR}/results.jsonl"
QUERIES_PATH = f"{OUT_DIR}/queries.jsonl"
CACHE_PATH = f"{OUT_DIR}/probe_feats.npz"
print("Folder:", OUT_DIR)

## 1. Load the 800-query dataset

In [ ]:
res = pd.read_json(RESULTS_PATH, lines=True)
qs = pd.read_json(QUERIES_PATH, lines=True).set_index("qid")
ok    = res.pivot(index="qid", columns="tier", values="correct")[ORDER].astype(float)
E_raw = res.pivot(index="qid", columns="tier", values="energy_j")[ORDER]
E_net = res.pivot(index="qid", columns="tier", values="energy_net_j")[ORDER]
complete = ok.dropna().index.intersection(E_raw.dropna().index).intersection(E_net.dropna().index)
ok, E_raw, E_net = ok.loc[complete], E_raw.loc[complete], E_net.loc[complete]
data = qs.loc[complete].copy()
qids = list(data.index)
ok_a, Er_a, En_a = ok.values, E_raw.values, E_net.values
ds = data["dataset"].values
texts = data["prompt"].tolist()
print(len(data), "queries;", pd.Series(ds).value_counts().to_dict())

## 2. Extract hidden-state features with the small model (GPU, about 5-10 minutes)
For each query we take the hidden state of the **last prompt token** at three depths of Qwen2.5-0.5B. Only the prompt is processed (no answer is generated).
GPU energy of this whole pass is measured, giving the **router overhead per query**.

In [ ]:
if os.path.exists(CACHE_PATH):
    z = np.load(CACHE_PATH, allow_pickle=True)
    cached_q = list(z["qids"])
    pos = {q: i for i, q in enumerate(cached_q)}
    LAYERS = [int(x) for x in z["layers"]]
    FEAT = {l: z[f"F{l}"][[pos[q] for q in qids]] for l in LAYERS}
    OVER_RAW, OVER_NET = float(z["over_raw"]), float(z["over_net"])
    print("Loaded cached features. Layers:", LAYERS)
else:
    import torch, pynvml
    from transformers import AutoTokenizer, AutoModel
    pynvml.nvmlInit(); H = pynvml.nvmlDeviceGetHandleByIndex(0)
    watts = lambda: pynvml.nvmlDeviceGetPowerUsage(H) / 1000.0

    class PowerMeter:
        def __init__(self, interval=0.02): self.interval, self.s, self._stop = interval, [], threading.Event()
        def _run(self):
            while not self._stop.is_set():
                self.s.append((time.perf_counter(), watts())); time.sleep(self.interval)
        def __enter__(self):
            self.s = []; self._stop.clear(); self._t = threading.Thread(target=self._run, daemon=True); self._t.start(); return self
        def __exit__(self, *a):
            self._stop.set(); self._t.join()
            self.energy_j = sum(0.5 * (w0 + w1) * (t1 - t0) for (t0, w0), (t1, w1) in zip(self.s, self.s[1:]))

    tok = AutoTokenizer.from_pretrained(PROBE_MODEL)
    tok.padding_side = "left"; tok.truncation_side = "left"
    model = AutoModel.from_pretrained(PROBE_MODEL, torch_dtype=torch.float16, device_map="cuda").eval()
    n_layers = model.config.num_hidden_layers
    LAYERS = sorted({n_layers // 2, (3 * n_layers) // 4, n_layers})
    prompts = [tok.apply_chat_template([{"role": "user", "content": t}], tokenize=False, add_generation_prompt=True)
               for t in texts]

    def run_batch(batch):
        enc = tok(batch, return_tensors="pt", padding=True, truncation=True, max_length=768).to("cuda")
        with torch.no_grad():
            out = model(**enc, output_hidden_states=True)
        return {l: out.hidden_states[l][:, -1, :].float().cpu().numpy() for l in LAYERS}

    run_batch(prompts[:BATCH]); torch.cuda.synchronize()                  # warm-up
    idle = np.mean([ (watts(), time.sleep(0.05))[0] for _ in range(60)])

    feats = {l: [] for l in LAYERS}
    with PowerMeter() as pm:
        t0 = time.perf_counter()
        for i in range(0, len(prompts), BATCH):
            o = run_batch(prompts[i:i + BATCH])
            for l in LAYERS: feats[l].append(o[l])
        torch.cuda.synchronize()
        elapsed = time.perf_counter() - t0
    FEAT = {l: np.vstack(feats[l]) for l in LAYERS}
    OVER_RAW = pm.energy_j / len(prompts)
    OVER_NET = max(pm.energy_j - idle * elapsed, 0.0) / len(prompts)
    np.savez(CACHE_PATH, qids=np.array(qids), layers=np.array(LAYERS),
             over_raw=OVER_RAW, over_net=OVER_NET, **{f"F{l}": FEAT[l] for l in LAYERS})
    print(f"Extraction took {elapsed:.0f} s")
    del model; torch.cuda.empty_cache()

print(f"Router overhead per query: {OVER_RAW:.2f} J raw, {OVER_NET:.2f} J net "
      f"(compare: a GSM8K answer costs about 400-900 J)")
print({l: FEAT[l].shape for l in LAYERS})

## 3. Can the features predict which tiers answer correctly? (GSM8K only: it holds about 98% of the energy)
AUC from 5-fold cross-validation (averaged over 3 shuffles). 0.5 = no signal.

In [ ]:
g = np.where(ds == "gsm8k")[0]
ng = len(g)
yg, Er_g, En_g = ok_a[g], Er_a[g], En_a[g]

KW = ["percent", "%", "each", "every", "per ", "twice", "half", "third", "double", "triple", "times",
      "more than", "less than", "remaining", "left", "total", "average", "fraction", "ratio",
      "week", "hour", "minute", "dollar", "$"]
def gfeat(t):
    body = t.split("\n\nSolve")[0]
    nums = re.findall(r"\d+(?:\.\d+)?", body.replace(",", ""))
    vals = [float(x) for x in nums] or [0.0]
    sents = [s for s in re.split(r"[.?!]\s", body) if s.strip()]
    low = body.lower()
    return [np.log1p(len(body)), np.log1p(len(body.split())), len(nums), len(sents),
            np.log1p(max(vals)), float(any("." in x for x in nums)), len(set(nums))] + [low.count(k) for k in KW]
G = np.array([gfeat(texts[i]) for i in g])

def cv_probs(F, C, seeds=(0, 1, 2)):
    P = np.zeros((ng, 3))
    for s in seeds:
        for tr, te in KFold(5, shuffle=True, random_state=s).split(F):
            for t in range(3):
                m = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=3000))
                P[te, t] += m.fit(F[tr], yg[tr, t]).predict_proba(F[te])[:, 1]
    return P / len(seeds)

sets = {"simple text features": (G, 0.3, 0.0, 0.0)}
for l in LAYERS:
    sets[f"probe layer {l}"] = (FEAT[l][g], 0.003, OVER_RAW, OVER_NET)
    sets[f"probe layer {l} + simple"] = (np.hstack([FEAT[l][g], G]), 0.003, OVER_RAW, OVER_NET)

PROBS, rows = {}, []
for name, (F, C, o_r, o_n) in sets.items():
    P = cv_probs(F, C)
    PROBS[name] = P
    rows.append({"features": name, **{tn: round(float(roc_auc_score(yg[:, t], P[:, t])), 3) for t, tn in enumerate(ORDER)}})
auc_tab = pd.DataFrame(rows).set_index("features")
auc_tab["mean"] = auc_tab.mean(axis=1).round(3)
print("AUC of predicting 'tier answers correctly' on GSM8K:")
print(auc_tab)
best = auc_tab.drop(index="simple text features")["mean"].idxmax()
print("\nBest probe feature set (by mean AUC):", best)

## 4. Route with the best probe and compare with static policies (overhead included)
Policy: pick the tier maximising `P(correct) - lambda * relative energy`. The probe's energy is **added to every routed query**.

In [ ]:
E_rel = Er_g.mean(0) / Er_g.mean(0).max()
ids = np.arange(ng)
LAMS = np.linspace(0, 1.5, 151)
pairs = [(0, 1), (1, 2), (0, 2)]
def choose(P, lam): return (P - lam * E_rel).argmax(1)

def curve(P, o_r, o_n):
    out = []
    for l in LAMS:
        ch = choose(P, l)
        out.append((yg[ids, ch].mean(), Er_g[ids, ch].mean() + o_r, En_g[ids, ch].mean() + o_n))
    return np.array(out)

pts = np.array([[yg[:, t].mean(), Er_g[:, t].mean(), En_g[:, t].mean()] for t in range(3)])
static = np.vstack([(1 - w) * pts[a] + w * pts[b] for a, b in pairs for w in np.linspace(0, 1, 101)])
acc_l, Er_l, En_l = pts[2]

def saving(cv, r, col):
    base = Er_l if col == 1 else En_l
    m = cv[:, 0] >= r * acc_l
    return np.nan if not m.any() else round(100 * (1 - cv[m, col].min() / base), 1)

rows = []
for r in [0.98, 0.95, 0.90]:
    row = {"accuracy kept": f"{int(r*100)}%"}
    for name in ["simple text features", best]:
        _, _, o_r, o_n = sets[name]
        cv = curve(PROBS[name], o_r, o_n)
        row[f"{name} raw %"] = saving(cv, r, 1); row[f"{name} net %"] = saving(cv, r, 2)
    row["best static raw %"] = saving(static, r, 1); row["best static net %"] = saving(static, r, 2)
    rows.append(row)
print("GSM8K: energy saved vs always-large (router overhead included)")
print(pd.DataFrame(rows).set_index("accuracy kept").T)

print("\nAlways-tier on GSM8K (acc, raw J):", {tn: (round(float(pts[t, 0]), 3), round(float(pts[t, 1]), 0)) for t, tn in enumerate(ORDER)})

## 5. How sure are we? Bootstrap confidence intervals
Resamples the queries 300 times and recomputes (router saving - best static saving). If the interval includes 0, the router is **not** reliably better than the best static policy.

In [ ]:
def boot(P, o_r, B=300, retains=(0.98, 0.95, 0.90), seed=0):
    CH = np.array([choose(P, l) for l in LAMS])              # (lambdas, queries)
    rng = np.random.default_rng(seed)
    diffs = {r: [] for r in retains}; rs = {r: [] for r in retains}; ss = {r: [] for r in retains}
    for _ in range(B):
        b = rng.integers(0, ng, ng)
        acc_t, e_t = yg[b].mean(0), Er_g[b].mean(0)
        acc_l_b, e_l_b = acc_t[2], e_t[2]
        racc = yg[b[None, :], CH[:, b]].mean(1)
        re_ = Er_g[b[None, :], CH[:, b]].mean(1) + o_r
        p2 = np.stack([acc_t, e_t], 1)
        st = np.vstack([(1 - w) * p2[a] + w * p2[c] for a, c in pairs for w in np.linspace(0, 1, 51)])
        for r in retains:
            def sv(acc, e):
                m = acc >= r * acc_l_b
                return np.nan if not m.any() else 100 * (1 - e[m].min() / e_l_b)
            rv, sv_ = sv(racc, re_), sv(st[:, 0], st[:, 1])
            rs[r].append(rv); ss[r].append(sv_); diffs[r].append(rv - sv_)
    rows = []
    for r in retains:
        d = np.array(diffs[r]); d = d[~np.isnan(d)]
        rows.append({"accuracy kept": f"{int(r*100)}%",
                     "router saving %": round(float(np.nanmean(rs[r])), 1),
                     "best static %": round(float(np.nanmean(ss[r])), 1),
                     "difference (95% CI)": f"{d.mean():+.1f} [{np.percentile(d, 2.5):+.1f}, {np.percentile(d, 97.5):+.1f}]" if len(d) else "n/a",
                     "router better in % of resamples": round(100 * float((d > 0).mean()), 0) if len(d) else np.nan})
    return pd.DataFrame(rows).set_index("accuracy kept")

for name in ["simple text features", best]:
    print(f"\n=== {name} (raw energy, overhead included) ===")
    print(boot(PROBS[name], sets[name][2]))

## How to read the result
- **Difference interval includes 0 (or the router is better in fewer than about 95% of resamples):** the router is *not* reliably better than the best static policy. Report this honestly.
- **Interval clearly above 0 at 95% and 98% accuracy:** you have a real, if modest, router win. Report the saving with its confidence interval.
- The probe's overhead is shown above; if it is more than 1-2% of an answer's energy, mention it in the limitations.
- **Next:** stop experimenting. Update the report with the 800-query numbers, add the confidence intervals, and build the demo.